# Convex Optimization

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/optimization-ml/02-convex-optimization

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
plt.rcParams['grid.color'] = '#2e3347'

BRAND  = '#818cf8'
TEAL   = '#14b8a6'
YELLOW = '#f59e0b'
ROSE   = '#f43f5e'

rng = np.random.default_rng(42)

## Intuition — why convexity is the dividing line

A **convex** loss has a single bowl: every local minimum is *the* global minimum, so gradient
descent from *any* starting point reaches the same answer, and you can even certify optimality.
This is the friendly world of linear/logistic regression and SVMs. A **non-convex** loss (any
neural network) has many valleys, saddles, and symmetries, so where you end up depends on where
you start and optimization only promises a *local* minimum. Knowing which regime you're in tells
you whether "the optimizer converged" means "we found the best model" (convex) or just "we found
*a* decent model" (non-convex). This notebook makes the boundary concrete.

## 1 — Visualising Convexity

A function is **convex** when any chord between two points lies *above* (or on) the curve.
Equivalently, $f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda)f(y)$ for all $\lambda \in [0,1]$.

We draw four functions on $[-2, 2]$ and, for each, mark the chord from $x=-1$ to $x=1$.

In [ ]:
x = np.linspace(-2, 2, 400)

funcs = [
    (lambda t: t**2,        r'$f(x)=x^2$',       TEAL,   True),
    (lambda t: np.abs(t),   r'$f(x)=|x|$',       BRAND,  True),
    (lambda t: -t**2,       r'$f(x)=-x^2$',      ROSE,   False),
    (lambda t: t**3,        r'$f(x)=x^3$',       YELLOW, False),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (f, label, color, is_convex) in zip(axes, funcs):
    ax.plot(x, f(x), color=color, linewidth=2, label=label)

    # Chord from x=-1 to x=1
    x0, x1 = -1.0, 1.0
    chord_x = np.array([x0, x1])
    chord_y = np.array([f(x0), f(x1)])
    ax.plot(chord_x, chord_y, '--', color='white', linewidth=1.5, alpha=0.8, label='chord')
    ax.scatter(chord_x, chord_y, color='white', zorder=5, s=40)

    verdict = 'CONVEX ✓' if is_convex else 'NOT convex ✗'
    v_color = TEAL if is_convex else ROSE
    ax.set_title(f'{label}\n{verdict}', color=v_color, fontsize=11)
    ax.set_xlim(-2.2, 2.2)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='#2e3347', linewidth=0.8)

plt.suptitle('Convexity: chord above curve = convex', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** the test is purely visual — for `x²` and `|x|` the dashed chord sits **on
or above** the curve everywhere (convex ✓); for `−x²` and `x³` the chord dips **below** somewhere
(not convex ✗). Convexity is exactly this "chord never goes below the curve" property, and it's
what guarantees a single global minimum.

## 2 — Jensen's Inequality

For a **convex** function $f$:

$$f\!\left(\mathbb{E}[X]\right) \leq \mathbb{E}[f(X)]$$

We verify this numerically for $f(x) = e^x$ and $X \sim \mathcal{N}(0, 1)$.
The moment-generating function gives $\mathbb{E}[e^X] = e^{\sigma^2/2} = e^{0.5} \approx 1.649$,
while $e^{\mathbb{E}[X]} = e^0 = 1$, so Jensen predicts $1.649 \geq 1$.

In [ ]:
samples = rng.standard_normal(10_000)

f_of_mean = np.exp(samples.mean())          # f(E[X])
mean_of_f = np.exp(samples).mean()          # E[f(X)]

print(f'f(E[X])  = exp(E[X]) = exp({samples.mean():.4f}) = {f_of_mean:.4f}')
print(f'E[f(X)]  = E[exp(X)]               = {mean_of_f:.4f}')
print(f'Jensen gap E[f(X)] - f(E[X])       = {mean_of_f - f_of_mean:.4f}  (should be >= 0)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(samples, bins=80, color=BRAND, alpha=0.5, density=True, label='samples $X$')
ax.axvline(samples.mean(), color=TEAL, linewidth=2, label=f'$\\mathbb{{E}}[X]={samples.mean():.3f}$')

ax2 = ax.twinx()
ax2.tick_params(colors='#94a3b8')
ax2.set_ylabel('$e^x$', color='#94a3b8')
xs = np.linspace(-3.5, 3.5, 300)
ax2.plot(xs, np.exp(xs), color=YELLOW, linewidth=2, label='$f(x)=e^x$')
ax2.scatter([samples.mean()], [f_of_mean], color=TEAL, zorder=6, s=80,
            label=f'$f(E[X])={f_of_mean:.3f}$')
ax2.scatter([samples.mean()], [mean_of_f], color=ROSE, zorder=6, s=80,
            label=f'$E[f(X)]={mean_of_f:.3f}$')
ax2.annotate('', xy=(samples.mean(), mean_of_f), xytext=(samples.mean(), f_of_mean),
             arrowprops=dict(arrowstyle='<->', color='white', lw=1.5))
ax2.text(samples.mean() + 0.1, (f_of_mean + mean_of_f) / 2, 'Jensen gap', color='white', fontsize=9)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
ax.set_title("Jensen's Inequality: $f(\\mathbb{E}[X]) \\leq \\mathbb{E}[f(X)]$ for convex $f$", color='white')
plt.tight_layout()
plt.show()

**What to notice:** the empirical **Jensen gap** `E[f(X)] − f(E[X])` comes out positive
(~0.65 for `eˣ` on a standard normal), matching the theoretical `e^{0.5} − 1`. For a convex `f`,
averaging *after* applying `f` always overshoots applying `f` to the average. Jensen's inequality
underpins the ELBO in variational inference and many loss bounds.

## 3 — Logistic Regression Loss is Convex

For a 1-D dataset, the cross-entropy loss as a function of scalar weight $w$ is:

$$\mathcal{L}(w) = -\frac{1}{n}\sum_i \bigl[y_i \log \sigma(w x_i) + (1-y_i)\log(1-\sigma(w x_i))\bigr]$$

Because the Hessian is $\sum_i \sigma(wx_i)(1-\sigma(wx_i))x_i^2 \geq 0$, this is **strictly convex** in $w$ —
there is exactly one global minimum.

In [ ]:
rng2 = np.random.default_rng(7)
n = 60
X1d = rng2.standard_normal(n)
y1d = (X1d + rng2.standard_normal(n) * 0.5 > 0).astype(float)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def cross_entropy(w, X, y, eps=1e-9):
    p = sigmoid(w * X)
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

w_grid = np.linspace(-6, 6, 400)
loss_vals = [cross_entropy(w, X1d, y1d) for w in w_grid]

w_opt = w_grid[np.argmin(loss_vals)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(w_grid, loss_vals, color=BRAND, linewidth=2, label='Cross-entropy loss')
ax.axvline(w_opt, color=TEAL, linestyle='--', linewidth=1.5,
           label=f'Global minimum $w^*\\approx{w_opt:.2f}$')
ax.scatter([w_opt], [min(loss_vals)], color=TEAL, zorder=6, s=80)
ax.set_xlabel('Weight $w$')
ax.set_ylabel('Loss $\\mathcal{L}(w)$')
ax.set_title('Logistic regression loss — single smooth bowl (convex)', color='white')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Loss is convex: second finite-diff curvature always >= 0?',
      np.all(np.diff(loss_vals, 2) >= -1e-6))

**What to notice:** the logistic (cross-entropy) loss is a single smooth **bowl** — the
finite-difference curvature is `≥ 0` everywhere, confirming convexity. There's exactly one
global minimum `w*`, so any optimizer finds the same weights. This is why logistic regression is
reliable in a way deep nets are not.

## 5. The library way — convexity means the optimizer can't get it wrong

Because the logistic loss is convex, a black-box optimizer reaches the global minimum from any
start. `scipy.optimize.minimize_scalar` finds it in one call; the cell asserts it matches the
grid-search minimum `w_opt`.

In [ ]:
from scipy.optimize import minimize_scalar

res = minimize_scalar(lambda w: cross_entropy(w, X1d, y1d))   # global for a convex 1-D loss

# cross-check against a fine grid over a wide range (the true min sits past the ±6 plot window)
wide = np.linspace(-15, 15, 60001)
losses = np.array([cross_entropy(w, X1d, y1d) for w in wide])
grid_min_w = wide[losses.argmin()]

print(f'scipy optimum      w = {res.x:.4f}')
print(f'fine-grid optimum  w = {grid_min_w:.4f}')
assert abs(res.x - grid_min_w) < 0.05, "scipy must find the same global minimum"
print('\nconvex loss -> a single global minimum, and the optimizer finds it ✓')

**What to notice:** scipy's Brent optimizer and the fine grid agree on the single global
minimum (out near `w ≈ 6.8`, past the plot window — this data is nearly separable, so the
weight wants to grow). Because the loss is convex there's only *one* minimum to find, so the
optimizer can't get stuck. Contrast that with the next section.

## 4 — Non-convex Neural Net Loss

A two-neuron network $f(x; a, b) = \sigma(ax) + \sigma(bx)$ has **permutation symmetry**: swapping
$a \leftrightarrow b$ gives the same output. The loss landscape therefore has (at least) two equivalent
global minima — breaking the single-bowl guarantee of convex problems.

We fit the squared loss $\mathcal{L}(a, b)$ on a small dataset and visualise it as a contour map.

In [ ]:
rng3 = np.random.default_rng(0)
X_nc = rng3.uniform(-2, 2, 20)
# target: sum of two sigmoids with a=2, b=-1
y_nc = sigmoid(2.0 * X_nc) + sigmoid(-1.0 * X_nc) + rng3.standard_normal(20) * 0.05

def two_neuron_loss(a, b):
    pred = sigmoid(a * X_nc) + sigmoid(b * X_nc)
    return np.mean((pred - y_nc) ** 2)

grid = np.linspace(-4, 4, 150)
A, B = np.meshgrid(grid, grid)
Z = np.vectorize(two_neuron_loss)(A, B)

fig, ax = plt.subplots(figsize=(7, 6))
cf = ax.contourf(A, B, Z, levels=40, cmap='magma')
cs = ax.contour(A, B, Z, levels=15, colors='white', alpha=0.25, linewidths=0.6)
plt.colorbar(cf, ax=ax, label='MSE loss')

# Mark the two symmetric global minima
ax.scatter([2.0], [-1.0], color=TEAL, s=100, zorder=6, label='$(a^*, b^*)=(2,-1)$')
ax.scatter([-1.0], [2.0], color=YELLOW, s=100, zorder=6, label='$(a^*, b^*)=(-1, 2)$ \u2014 same loss!')
ax.plot([2, -1], [-1, 2], '--', color='white', alpha=0.5, linewidth=1.2, label='symmetry axis $a=b$')

ax.set_xlabel('$a$ (weight of neuron 1)')
ax.set_ylabel('$b$ (weight of neuron 2)')
ax.set_title('2-neuron net: permutation symmetry → two global minima', color='white')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**What to notice:** the two-neuron loss has **two** equally-good minima — `(2,−1)` and
`(−1,2)` — because swapping the neurons leaves the output unchanged (permutation symmetry). The
single-bowl guarantee is gone; which minimum you reach depends on initialization. Every neural
network has this, multiplied across millions of symmetric weights.

## 6. Gotchas & tradeoffs

- **Convex is the exception, not the rule.** Linear/logistic regression, SVMs, and LASSO are
  convex; **every neural network is non-convex** (symmetries, saddles, many minima).
- **Non-convex ⇒ initialization and seed matter.** Different starts → different solutions, so
  results aren't reproducible without fixing the seed.
- **"Converged" ≠ "optimal"** for non-convex losses — you found a *local* minimum, and there's no
  cheap certificate that it's global.
- **Convexity is fragile.** Adding a hidden layer, a non-monotone activation, or certain
  regularizers can destroy it.

In [ ]:
# Non-convex: gradient descent lands in DIFFERENT minima depending on where it starts
def gd_two_neuron(a0, b0, lr=0.5, steps=3000, eps=1e-4):
    a, b = a0, b0
    for _ in range(steps):
        ga = (two_neuron_loss(a + eps, b) - two_neuron_loss(a - eps, b)) / (2 * eps)
        gb = (two_neuron_loss(a, b + eps) - two_neuron_loss(a, b - eps)) / (2 * eps)
        a -= lr * ga; b -= lr * gb
    return float(round(a, 2)), float(round(b, 2))

print('start (3, -2) -> minimum', gd_two_neuron(3.0, -2.0))
print('start (-2, 3) -> minimum', gd_two_neuron(-2.0, 3.0))
print('\nsame loss surface, two different answers -> the hallmark of non-convexity')

**What to notice:** starting near `(3,−2)` converges toward the `(2,−1)` basin, but starting
near `(−2,3)` converges to its mirror image `(−1,2)` — a *different* set of weights with the
same loss. On a convex problem both starts would agree; here initialization decides which
symmetric minimum you land in, which is why deep-learning results always pin the random seed.

## Key takeaways

- **Convex** = chord-above-curve = one global minimum; gradient descent finds it from any start
  (verified with `scipy`).
- **Jensen's inequality** (`E[f(X)] ≥ f(E[X])` for convex `f`) has a positive gap that vanishes
  only for linear `f`.
- **Logistic/linear/SVM losses are convex**; **neural networks are non-convex** (permutation
  symmetry → many equivalent minima).
- Non-convex ⇒ **init/seed matter**, "converged" only means a **local** minimum, and convexity is
  easily lost by adding depth.

**Next:** [Constrained Optimization](https://ml-viz-ruby.vercel.app/courses/optimization-ml/03-constrained-optimization).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set,
and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently
when your answer is right.

### Exercise 1 — `is_convex_on_grid(f, x_min, x_max, n=200)`

Test convexity numerically using the **midpoint rule**: a function is convex if

$$f\!\left(\frac{x+y}{2}\right) \leq \frac{f(x)+f(y)}{2} \quad \text{for all pairs } x, y$$

Build an $n$-point grid, then check the midpoint rule for all $\binom{n}{2}$ pairs.
Return `True` if the rule holds for every pair (with a small numerical tolerance `1e-9`),
`False` otherwise.

In [ ]:
def is_convex_on_grid(f, x_min, x_max, n=200):
    """
    Returns True if f satisfies the midpoint convexity condition
    f((x+y)/2) <= (f(x)+f(y))/2 for all pairs in an n-point grid.
    """
    grid = np.linspace(x_min, x_max, n)
    fvals = f(grid)

    # TODO(you): for every pair of indices i < j, compute:
    #   midpoint = (grid[i] + grid[j]) / 2
    #   lhs = f(midpoint)
    #   rhs = (fvals[i] + fvals[j]) / 2
    #   check lhs <= rhs + 1e-9
    # Hint: you can vectorise with meshgrid or just use a double loop.

    return ...  # True or False

In [ ]:
assert is_convex_on_grid(lambda x: x**2,      -5, 5) is True,  'x^2 is convex'
assert is_convex_on_grid(lambda x: -x**2,     -5, 5) is False, '-x^2 is concave'
assert is_convex_on_grid(lambda x: np.abs(x), -5, 5) is True,  '|x| is convex'
print('\u2705 Exercise 1 passed')

<details>
<summary>💡 Show solution</summary>

```python
def is_convex_on_grid(f, x_min, x_max, n=200):
    grid = np.linspace(x_min, x_max, n)
    fvals = f(grid)
    i_idx, j_idx = np.triu_indices(n, k=1)
    midpoints = (grid[i_idx] + grid[j_idx]) / 2.0
    lhs = f(midpoints)
    rhs = (fvals[i_idx] + fvals[j_idx]) / 2.0
    return bool(np.all(lhs <= rhs + 1e-9))
```

</details>

### Exercise 2 — `jensen_gap(samples, f)`

Jensen's inequality says $\mathbb{E}[f(X)] \geq f(\mathbb{E}[X])$ for convex $f$.
The **Jensen gap** is the difference:

$$\text{gap} = \mathbb{E}[f(X)] - f(\mathbb{E}[X]) \geq 0$$

Implement `jensen_gap(samples, f)` that takes an array of samples and a function,
and returns the empirical Jensen gap. Then verify:
- Gap $\geq 0$ for convex functions ($e^x$, $x^2$, $|x|$)
- Gap $\approx 0$ for linear $f(x) = x$ (Jensen is tight for linear functions)

In [ ]:
def jensen_gap(samples, f):
    """
    Returns E[f(X)] - f(E[X]).
    Should be >= 0 for convex f, ~0 for linear f.
    """
    samples = np.asarray(samples, dtype=float)
    # TODO(you): compute E[f(X)] and f(E[X]) and return their difference
    return ...

In [ ]:
samps = rng.standard_normal(50_000)

assert jensen_gap(samps, np.exp)          >= 0,    'exp is convex -> gap >= 0'
assert jensen_gap(samps, lambda x: x**2) >= 0,    'x^2 is convex -> gap >= 0'
assert jensen_gap(samps, np.abs)         >= 0,    '|x| is convex -> gap >= 0'
assert abs(jensen_gap(samps, lambda x: x)) < 0.01, 'linear -> Jensen tight, gap ~ 0'
print('\u2705 Exercise 2 passed')

<details>
<summary>💡 Show solution</summary>

```python
def jensen_gap(samples, f):
    samples = np.asarray(samples, dtype=float)
    mean_of_f = np.mean(f(samples))
    f_of_mean = f(np.mean(samples))
    return float(mean_of_f - f_of_mean)
```

</details>